# 06 — Full-Universe Compatibility, Clustering & Visualization

**CRISP-DM phase:** Modelling (clustering) / Evaluation (cluster validity) / Deployment prep (figures + data for the app).

Three things happen here, all feeding the Streamlit app:
1. Compute every player-season's compatibility score against all three managers (not just the players who actually played under them) -- this is what lets the app answer "which manager does THIS player fit?" for any player in the top-5 leagues.
2. Cluster the full player universe (k-means, validated with silhouette/Davies-Bouldin/Calinski-Harabasz per brief SS31 -- the clustering analysis the project was missing) to identify data-driven playing-style groups, independent of the three managers.
3. Build the visualizations the project was also missing: manager profile comparisons, a 2D PCA projection of the player universe coloured by cluster and by best-fit manager, and compatibility-score distributions.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

from data_sources import DATA_PROCESSED
from football_features import DIMENSION_WEIGHTS
from compatibility import compute_compatibility, rescale_to_index

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

FIG_DIR = Path.cwd().parent / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
dim_cols = list(DIMENSION_WEIGHTS.keys())

universe = pd.read_parquet(DATA_PROCESSED / "player_universe.parquet")
manager_profiles = pd.read_parquet(DATA_PROCESSED / "manager_profiles.parquet")
print(universe.shape)

(11132, 24)


## 1. Full-universe compatibility scores

In [2]:
universe_key_cols = universe.rename(columns={"Squad": "team"}).assign(manager="n/a")
universe_compat = compute_compatibility(universe_key_cols, manager_profiles, dim_cols, min_dims_required=4)
universe_compat["compatibility_index_0_100"] = universe_compat.groupby("scored_against_manager")["compatibility_similarity"].transform(rescale_to_index)
print(universe_compat.shape)

best = universe_compat.loc[universe_compat.groupby(["Player", "Season_End_Year", "team"])["compatibility_similarity"].idxmax()]
best = best.rename(columns={"scored_against_manager": "best_fit_manager", "compatibility_similarity": "best_fit_similarity"})
universe_with_fit = universe.merge(
    best[["Player", "Season_End_Year", "team", "best_fit_manager", "best_fit_similarity"]].rename(columns={"team": "Squad"}),
    on=["Player", "Season_End_Year", "Squad"], how="left",
)
universe_with_fit.best_fit_manager.value_counts()

(33396, 18)


best_fit_manager
Roberto De Zerbi    6032
Jürgen Klopp        2630
Pep Guardiola       2470
Name: count, dtype: int64

## 2. Clustering the player universe (brief SS28-31)

K-means over the 7 tactical dimensions, position-agnostic (positions already influence the dimension scores via the position-group z-scoring in notebook 03/05, so this clusters PLAYING STYLE, not literal position). Missing dimensions are mean-imputed for clustering only (documented, not silently done) -- clustering algorithms need complete rows, unlike the NaN-aware similarity method used elsewhere in this project.

In [3]:
X = universe[dim_cols].copy()
impute_rate = X.isna().mean()
print("Share of rows imputed (mean-filled) per dimension for clustering purposes only:")
print(impute_rate.round(4))
X = X.fillna(X.mean())
Xs = StandardScaler().fit_transform(X)

Share of rows imputed (mean-filled) per dimension for clustering purposes only:
TD1_buildup_directness       0.0002
TD3_progression_passing      0.0002
TD4_progression_carrying     0.1041
TD6_chance_creation          0.0002
TD7_box_presence             0.0002
TD9_pressing_intensity       0.0000
TD11_individual_defending    0.0002
dtype: float64


In [4]:
results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    labels = km.labels_
    results.append({
        "k": k,
        "silhouette": silhouette_score(Xs, labels),
        "davies_bouldin": davies_bouldin_score(Xs, labels),
        "calinski_harabasz": calinski_harabasz_score(Xs, labels),
        "inertia": km.inertia_,
    })
validity = pd.DataFrame(results)
validity

,k,silhouette,davies_bouldin,calinski_harabasz,inertia
0,2,0.190750,1.827671,2781.784683,62342.471575
1,3,0.182123,1.662572,2433.082336,54217.516773
2,4,0.132914,1.726605,2070.810939,50006.845536
3,5,0.133069,1.619243,1876.672246,46532.039769
4,6,0.132876,1.651494,1707.333718,44092.851655
5,7,0.128243,1.699360,1586.547064,41992.592764
6,8,0.129755,1.654743,1499.439383,40093.807880
7,9,0.121393,1.686733,1409.210996,38700.004716
8,10,0.122153,1.643284,1333.430205,37481.138701


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(validity.k, validity.silhouette, marker="o"); axes[0].set_title("Silhouette (higher better)"); axes[0].set_xlabel("k")
axes[1].plot(validity.k, validity.davies_bouldin, marker="o", color="orange"); axes[1].set_title("Davies-Bouldin (lower better)"); axes[1].set_xlabel("k")
axes[2].plot(validity.k, validity.calinski_harabasz, marker="o", color="green"); axes[2].set_title("Calinski-Harabasz (higher better)"); axes[2].set_xlabel("k")
plt.tight_layout()
plt.savefig(FIG_DIR / "cluster_validity_indices.png", dpi=120)
plt.show()

best_k = int(validity.loc[validity.silhouette.idxmax(), "k"])
print(f"Best k by silhouette: {best_k} (silhouette={validity.silhouette.max():.3f} -- note: values below ~0.25 indicate weak, overlapping clusters, consistent with the continuous, not sharply clustered, nature of football playing styles)")

Best k by silhouette: 2 (silhouette=0.191 -- note: values below ~0.25 indicate weak, overlapping clusters, consistent with the continuous, not sharply clustered, nature of football playing styles)


/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_12339/1357326556.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
FINAL_K = max(best_k, 5)  # at least 5 clusters for a useful archetype vocabulary in the app, even if silhouette peaks lower
km_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10).fit(Xs)
universe["cluster"] = km_final.labels_

cluster_profiles = universe.groupby("cluster")[dim_cols].mean().round(3)
cluster_profiles["n_players"] = universe.cluster.value_counts().sort_index()
cluster_profiles

,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD9_pressing_intensity,TD11_individual_defending,n_players
cluster,,,,,,,,
0,0.031,-0.011,-0.168,-0.192,-0.218,0.167,0.414,2025
1,-0.378,-0.336,-0.208,-0.114,0.332,-0.298,0.105,2160
2,0.063,0.031,-0.153,-0.137,-0.237,-0.246,-0.243,3168
3,-0.001,0.136,0.624,0.771,0.558,0.019,-0.169,1740
4,0.496,0.459,0.126,0.036,-0.223,0.261,-0.145,2039


## 3. Name the clusters from their profiles (football-language labelling)

Cluster centroids are translated into a short football label from their two highest-magnitude dimensions -- a lightweight, transparent version of the archetype-naming approach used in the literature (e.g. Baattite 2023, Plakias et al. 2025).

In [7]:
DIM_SHORT = {
    "TD1_buildup_directness": "build-up", "TD3_progression_passing": "progressive passing",
    "TD4_progression_carrying": "progressive carrying", "TD6_chance_creation": "chance creation",
    "TD7_box_presence": "box presence", "TD9_pressing_intensity": "pressing", "TD11_individual_defending": "defending",
}

def label_cluster(row):
    ranked = row[dim_cols].sort_values(ascending=False)
    top = DIM_SHORT[ranked.index[0]]
    second = DIM_SHORT[ranked.index[1]]
    return f"{top.title()} + {second.title()}"

cluster_labels = {c: label_cluster(cluster_profiles.loc[c]) for c in cluster_profiles.index}
universe["cluster_label"] = universe["cluster"].map(cluster_labels)
for c, lbl in cluster_labels.items():
    print(f"Cluster {c} ({(universe.cluster==c).sum()} players): {lbl}")

Cluster 0 (2025 players): Defending + Pressing
Cluster 1 (2160 players): Box Presence + Defending
Cluster 2 (3168 players): Build-Up + Progressive Passing
Cluster 3 (1740 players): Chance Creation + Progressive Carrying
Cluster 4 (2039 players): Build-Up + Progressive Passing


## 4. Visualizations

In [8]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(Xs)
universe["pca_x"], universe["pca_y"] = coords[:, 0], coords[:, 1]
print(f"Variance explained by 2D projection: {pca.explained_variance_ratio_.sum():.1%}")

fig, ax = plt.subplots(figsize=(9, 7))
for c in sorted(universe.cluster.unique()):
    sub = universe[universe.cluster == c]
    ax.scatter(sub.pca_x, sub.pca_y, s=8, alpha=0.5, label=f"{c}: {cluster_labels[c]}")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title("Player universe -- PCA projection coloured by cluster")
ax.legend(fontsize=7, loc="best")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_clusters.png", dpi=120)
plt.show()

Variance explained by 2D projection: 55.6%


/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_12339/3191356572.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
fig, ax = plt.subplots(figsize=(9, 5))
manager_profiles.set_index("manager")[dim_cols].T.plot(kind="bar", ax=ax)
ax.set_title("Manager tactical-dimension profiles")
ax.set_ylabel("Dimension score (z-scored vs league position baseline)")
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "manager_profiles_bar.png", dpi=120)
plt.show()

/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_12339/2515652121.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
import matplotlib.pyplot as plt

def radar_chart(ax, profiles, dims, labels, colors):
    angles = np.linspace(0, 2 * np.pi, len(dims), endpoint=False).tolist()
    angles += angles[:1]
    for prof, lbl, col in zip(profiles, labels, colors):
        vals = list(prof) + [prof[0]]
        ax.plot(angles, vals, label=lbl, color=col)
        ax.fill(angles, vals, alpha=0.08, color=col)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([DIM_SHORT[d] for d in dims], fontsize=8)

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
mp = manager_profiles.set_index("manager")
radar_chart(ax, [mp.loc[m, dim_cols].values for m in mp.index], dim_cols, mp.index.tolist(), ["#1f77b4", "#d62728", "#2ca02c"])
ax.set_title("Manager tactical profiles (radar)")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "manager_profiles_radar.png", dpi=120)
plt.show()

/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_12339/564064809.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
fig, ax = plt.subplots(figsize=(8, 5))
for m in universe_compat.scored_against_manager.unique():
    sub = universe_compat[universe_compat.scored_against_manager == m]
    ax.hist(sub.compatibility_similarity.dropna(), bins=40, alpha=0.5, label=m)
ax.set_xlabel("Compatibility similarity"); ax.set_ylabel("Player-seasons")
ax.set_title("Distribution of compatibility scores across the full player universe")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "compatibility_distributions.png", dpi=120)
plt.show()

/var/folders/jz/y19hl2rx6gz8fhcs7qnq2pvw0000gn/T/ipykernel_12339/2082524568.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Save everything the app needs

In [12]:
universe_final = universe.merge(
    best[["Player", "Season_End_Year", "team", "best_fit_manager", "best_fit_similarity"]].rename(columns={"team": "Squad"}),
    on=["Player", "Season_End_Year", "Squad"], how="left",
)
universe_final.to_parquet(DATA_PROCESSED / "player_universe.parquet", index=False)
universe_compat.to_parquet(DATA_PROCESSED / "universe_compatibility.parquet", index=False)
validity.to_csv(Path.cwd().parent / "outputs" / "tables" / "cluster_validity_indices.csv", index=False)
cluster_profiles.to_csv(Path.cwd().parent / "outputs" / "tables" / "cluster_profiles.csv")
pd.Series(cluster_labels, name="label").to_csv(Path.cwd().parent / "outputs" / "tables" / "cluster_labels.csv")
print("saved:", universe_final.shape, universe_compat.shape)

saved: (11132, 30) (33396, 18)
